# Self supervised learning use cases

To run on Kaggle with 2*T4 Gpus

### Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset, Subset
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# Ensure your ml_pipeline folder is accessible in the Kaggle working directory or sys.path
import sys
import os
# sys.path.append(os.path.abspath("/kaggle/input/your-dataset-name")) # Uncomment and adjust if loading from a Kaggle dataset

from ml_pipeline.pipelines_torch.vision_models import get_model
from ml_pipeline.pipelines_torch.ssl_algorithms import MeanTeacher
from ml_pipeline.pipelines_torch.models import TorchMLPClassifier
from ml_pipeline.pipelines_torch.ss_models import SemiSupervisedTabular

# Setup Device (Utilizing Kaggle's T4 GPUs)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### Vision Dataset Preparation (CIFAR 10)

In [ ]:
# Image transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load full CIFAR-10 training data
full_train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Create Semi-Supervised splits: 4000 labeled, 46000 unlabeled
num_labeled = 4000
indices = np.random.permutation(len(full_train_dataset))
labeled_indices = indices[:num_labeled]
unlabeled_indices = indices[num_labeled:]

labeled_dataset = Subset(full_train_dataset, labeled_indices)
# For unlabeled data, we simply ignore the targets during training
unlabeled_dataset = Subset(full_train_dataset, unlabeled_indices)

# Dataloaders
batch_size_l = 32
batch_size_u = 128 # Typically larger batch size for unlabeled

loader_l = DataLoader(labeled_dataset, batch_size=batch_size_l, shuffle=True, drop_last=True)
loader_u = DataLoader(unlabeled_dataset, batch_size=batch_size_u, shuffle=True, drop_last=True)
loader_test_vis = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f"Vision Labeled samples: {len(labeled_dataset)} | Unlabeled samples: {len(unlabeled_dataset)}")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

from ml_pipeline.pipelines_torch.benchmark import BenchmarkRunner
from ml_pipeline.pipelines_torch.vision_models import get_model
from ml_pipeline.pipelines_torch.ssl_algorithms import MeanTeacher, PseudoLabel, PiModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Load CIFAR-10 as Numpy arrays for BenchmarkRunner
print("Loading CIFAR-10 dataset...")
full_train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
X_vis = full_train_dataset.data
y_vis = np.array(full_train_dataset.targets)

# Preprocess: (N, H, W, C) -> (N, C, H, W) and scale to [-1, 1]
X_vis = np.transpose(X_vis, (0, 3, 1, 2)).astype(np.float32) / 255.0
X_vis = (X_vis - 0.5) / 0.5

# Subset dataset to 15,000 samples to keep notebook execution fast
X_vis, _, y_vis, _ = train_test_split(X_vis, y_vis, train_size=15000, stratify=y_vis, random_state=42)

# 2. Create an SSL Wrapper compatible with GeneralPipeline's custom fit interception
class SSLVisionWrapper(nn.Module):
    def __init__(self, ssl_algo="mean_teacher", num_classes=10, num_labeled=2000):
        super().__init__()
        base_model = get_model("simple_cnn", num_classes=num_classes)
        self.ssl_algo = ssl_algo
        self.num_labeled = num_labeled

        # Inject standard weak/strong augmentations directly
        def weak_aug(x): return x + torch.randn_like(x) * 0.05
        def strong_aug(x): return x + torch.randn_like(x) * 0.15

        if ssl_algo == "mean_teacher":
            self.ssl_model = MeanTeacher(base_model, ema_decay=0.999, weak=weak_aug, strong=strong_aug)
        elif ssl_algo == "pseudo_label":
            self.ssl_model = PseudoLabel(base_model, threshold=0.95, weak=weak_aug, strong=strong_aug)
        elif ssl_algo == "pi_model":
            self.ssl_model = PiModel(base_model, weak=weak_aug, strong=strong_aug)
        else:
            self.ssl_model = base_model # Pure supervised fallback

    def forward(self, x):
        return self.ssl_model(x)

    def fit(self, X, y, epochs=5, batch_size=32, learning_rate=1e-3, **kwargs):
        dev = next(self.parameters()).device

        # GeneralPipeline passes tensors if using mixed modes, cast them back to numpy for splitting
        if torch.is_tensor(X): X = X.cpu().numpy()
        if torch.is_tensor(y): y = y.cpu().numpy()

        # Simulate semi-supervised environment inside the custom fit logic
        X_l, X_u, y_l, y_u = train_test_split(X, y, train_size=self.num_labeled, stratify=y, random_state=42)

        l_loader = DataLoader(TensorDataset(torch.FloatTensor(X_l), torch.LongTensor(y_l)), batch_size=batch_size, shuffle=True, drop_last=True)

        if self.ssl_algo != "supervised":
            u_loader = DataLoader(TensorDataset(torch.FloatTensor(X_u), torch.LongTensor(y_u)), batch_size=batch_size*2, shuffle=True, drop_last=True)

        opt = torch.optim.Adam(self.parameters(), lr=learning_rate)
        self.train()
        global_step = 0
        history = []

        for epoch in range(epochs):
            if self.ssl_algo != "supervised": iter_u = iter(u_loader)
            total_loss = 0

            for batch_l in l_loader:
                bl = (batch_l[0].to(dev), batch_l[1].to(dev))
                opt.zero_grad()

                if self.ssl_algo != "supervised":
                    try:
                        batch_u = next(iter_u)
                    except StopIteration:
                        iter_u = iter(u_loader)
                        batch_u = next(iter_u)
                    bu = (batch_u[0].to(dev), None)

                    # Compute SSL wrapper loss
                    loss_sup, loss_unsup, _ = self.ssl_model.ssl_loss(bl, bu, global_step)
                    loss = loss_sup + loss_unsup
                else:
                    # Supervised objective
                    logits = self.ssl_model(bl[0])
                    loss = F.cross_entropy(logits, bl[1])

                loss.backward()
                opt.step()

                # Advance teacher EMA if appropriate
                if hasattr(self.ssl_model, 'update_teacher'):
                    self.ssl_model.update_teacher()

                total_loss += loss.item()
                global_step += 1

            epoch_loss = total_loss/len(l_loader)
            print(f"[{self.ssl_algo}] Vision Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f}")
            history.append({'epoch': epoch+1, 'loss': epoch_loss})

        return history

    def predict_proba(self, X):
        self.eval()
        with torch.no_grad():
            logits = self.forward(torch.FloatTensor(X).to(next(self.parameters()).device))
            return torch.softmax(logits, dim=1).cpu().numpy()

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

# 3. Configure and trigger the BenchmarkRunner
vision_models = [
    {"name": "Vision_Supervised_Baseline", "class": SSLVisionWrapper, "params": {"ssl_algo": "supervised"}},
    {"name": "Vision_MeanTeacher", "class": SSLVisionWrapper, "params": {"ssl_algo": "mean_teacher"}},
    {"name": "Vision_PseudoLabel", "class": SSLVisionWrapper, "params": {"ssl_algo": "pseudo_label"}},
    {"name": "Vision_PiModel", "class": SSLVisionWrapper, "params": {"ssl_algo": "pi_model"}}
]

runner_vis = BenchmarkRunner(
    model_configs=vision_models,
    augmentations=[None],
    task_type="classification",
    device=str(device),
    epochs=5,
    batch_size=32,
    use_kfold=False, # We override data management with custom fit anyway
    path_start="benchmark_vision_ssl"
)

print("\n--- Starting Vision SSL Benchmark ---")
runner_vis.run(X_vis, y_vis)

### Tabular dataset Preparation

In [ ]:
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import DataLoader, TensorDataset

print("Fetching Forest Covertype dataset (this may take a moment)...")
covtype = fetch_covtype()
X = covtype.data
# Labels are 1-7, PyTorch expects 0-6 for a 7-class problem
y = covtype.target - 1

# We will use a subset to keep training times reasonable for the notebook
X, _, y, _ = train_test_split(X, y, train_size=100000, stratify=y, random_state=42)

# Scale features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split: 20000 Test, 80000 Train/Unlabeled
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=20000, stratify=y, random_state=42
)

# From the 80000, keep only 500 as strictly labeled. The rest are unlabeled.
X_l, X_u, y_l, y_u = train_test_split(
    X_train_full, y_train_full, train_size=500, stratify=y_train_full, random_state=42
)

# Convert to PyTorch Tensors
tensor_X_l = torch.FloatTensor(X_l)
tensor_y_l = torch.LongTensor(y_l)
tensor_X_u = torch.FloatTensor(X_u)
tensor_y_u = torch.LongTensor(y_u) # Kept for structure, not used for SSL training
tensor_X_test = torch.FloatTensor(X_test)
tensor_y_test = torch.LongTensor(y_test)

tab_loader_l = DataLoader(TensorDataset(tensor_X_l, tensor_y_l), batch_size=32, shuffle=True, drop_last=True)
tab_loader_u = DataLoader(TensorDataset(tensor_X_u, tensor_y_u), batch_size=256, shuffle=True, drop_last=True)
tab_loader_test = DataLoader(TensorDataset(tensor_X_test, tensor_y_test), batch_size=512, shuffle=False)

print(f"Tabular Labeled: {len(X_l)} | Unlabeled: {len(X_u)} | Test: {len(X_test)}")
print(f"Features: {X.shape[1]} | Classes: {len(set(y))}")

In [ ]:
from sklearn.datasets import fetch_covtype
from sklearn.preprocessing import StandardScaler
from ml_pipeline.pipelines_torch.models import TorchMLPClassifier
from ml_pipeline.pipelines_torch.ss_models import SemiSupervisedTabular

print("Fetching Forest Covertype dataset (tabular)...")
covtype = fetch_covtype()
X_tab = covtype.data
y_tab = covtype.target - 1  # Standardize labels to 0-indexed PyTorch format

# Subset to keep execution times fast and scale features
X_tab, _, y_tab, _ = train_test_split(X_tab, y_tab, train_size=20000, stratify=y_tab, random_state=42)
X_tab = StandardScaler().fit_transform(X_tab)

# 1. Create a custom Tabular Wrapper targeting BenchmarkRunner interception
class SSLTabularWrapper(nn.Module):
    def __init__(self, ssl_algo="mean_teacher", input_dim=54, num_classes=7, num_labeled=500):
        super().__init__()
        self.base_model = TorchMLPClassifier(input_dim, [128, 64], num_classes, dropout=0.2, batchnorm=True)
        self.ssl_algo = ssl_algo
        self.num_labeled = num_labeled

        # Inject standard ss_models wrapper
        if ssl_algo != "supervised":
            self.ssl_model = SemiSupervisedTabular(
                self.base_model, num_classes,
                use_mean_teacher=(ssl_algo == "mean_teacher"),
                rampup=50
            )
        else:
            self.ssl_model = self.base_model

    def forward(self, x):
        return self.ssl_model(x)

    def fit(self, X, y, epochs=10, batch_size=32, learning_rate=1e-3, **kwargs):
        dev = next(self.parameters()).device
        if torch.is_tensor(X): X = X.cpu().numpy()
        if torch.is_tensor(y): y = y.cpu().numpy()

        # Split internally to respect the benchmark flow without changing Runner internals
        X_l, X_u, y_l, y_u = train_test_split(X, y, train_size=self.num_labeled, stratify=y, random_state=42)

        l_loader = DataLoader(TensorDataset(torch.FloatTensor(X_l), torch.LongTensor(y_l)), batch_size=batch_size, shuffle=True, drop_last=True)

        if self.ssl_algo != "supervised":
            u_loader = DataLoader(TensorDataset(torch.FloatTensor(X_u), torch.LongTensor(y_u)), batch_size=batch_size*4, shuffle=True, drop_last=True)

        opt = torch.optim.Adam(self.parameters(), lr=learning_rate)
        self.train()
        global_step = 0
        history = []

        for epoch in range(epochs):
            if self.ssl_algo != "supervised": iter_u = iter(u_loader)
            total_loss = 0

            for batch_l in l_loader:
                bl = (batch_l[0].to(dev), batch_l[1].to(dev))
                opt.zero_grad()

                if self.ssl_algo != "supervised":
                    try:
                        batch_u = next(iter_u)
                    except StopIteration:
                        iter_u = iter(u_loader)
                        batch_u = next(iter_u)
                    bu = (batch_u[0].to(dev), None)

                    # SSL step uses SemiSupervisedTabular class logic
                    loss, _ = self.ssl_model.step(bl, bu, global_step)
                else:
                    logits = self.ssl_model(bl[0])
                    loss = F.cross_entropy(logits, bl[1])

                loss.backward()
                opt.step()

                if hasattr(self.ssl_model, 'post_step'):
                    self.ssl_model.post_step()

                total_loss += loss.item()
                global_step += 1

            epoch_loss = total_loss/len(l_loader)
            print(f"[{self.ssl_algo}] Tabular Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f}")
            history.append({'epoch': epoch+1, 'loss': epoch_loss})

        return history

    def predict_proba(self, X):
        self.eval()
        with torch.no_grad():
            logits = self.forward(torch.FloatTensor(X).to(next(self.parameters()).device))
            return torch.softmax(logits, dim=1).cpu().numpy()

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

# 2. Configure and trigger the BenchmarkRunner
tabular_models = [
    {"name": "Tabular_Supervised_Baseline", "class": SSLTabularWrapper, "params": {"ssl_algo": "supervised"}},
    {"name": "Tabular_MeanTeacher", "class": SSLTabularWrapper, "params": {"ssl_algo": "mean_teacher"}},
    {"name": "Tabular_PseudoLabel", "class": SSLTabularWrapper, "params": {"ssl_algo": "pseudo_label"}},
]

runner_tab = BenchmarkRunner(
    model_configs=tabular_models,
    augmentations=[None],
    task_type="classification",
    device=str(device),
    epochs=10,
    batch_size=32,
    use_kfold=False,
    path_start="benchmark_tabular_ssl"
)

print("\n--- Starting Tabular SSL Benchmark ---")
runner_tab.run(X_tab, y_tab)